# AutoBI — Step 2: Sample Data Ingestion Walkthrough (SYNTHETIC)

This notebook demonstrates loading the three **SYNTHETIC** sample datasets (`sales`, `HR`, `healthcare`) through `core.ingestion` and staging them into Parquet + DuckDB via `core.storage`.

> **Note:** All datasets under `sample_data/` are explicitly **SYNTHETIC** test data.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "core").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import Settings
from core.ingestion import ingest_files
from core.storage import list_duckdb_tables, query_duckdb, write_staging_tables

sample_files = [
    PROJECT_ROOT / "sample_data" / "SYNTHETIC_sales.csv",
    PROJECT_ROOT / "sample_data" / "SYNTHETIC_hr.xlsx",
    PROJECT_ROOT / "sample_data" / "SYNTHETIC_healthcare.csv",
]
[p.name for p in sample_files]

In [ ]:
settings = Settings(_env_file=None, data_dir=PROJECT_ROOT / "data")
bundle, frames = ingest_files(sample_files, settings=settings)
staged_bundle = write_staging_tables(bundle, frames, settings=settings)

print(f"Batch ID: {staged_bundle.batch_id}")
print(f"Total rows: {staged_bundle.total_rows}, Total columns: {staged_bundle.total_columns}")
print(f"Load warnings: {staged_bundle.load_warnings}")
for tbl in staged_bundle.tables:
    print(f"  - {tbl.table_name}: {tbl.row_count} rows x {tbl.column_count} cols (header row {tbl.header_row_index})")

In [ ]:
assert staged_bundle.manifest_path is not None
manifest_data = json.loads(staged_bundle.manifest_path.read_text(encoding="utf-8"))
for item in manifest_data["files"]:
    print(item["filename"], "| size:", item["size_bytes"], "| sha256:", item["sha256"][:16] + "...")

In [ ]:
assert staged_bundle.duckdb_path is not None
registered_tables = list_duckdb_tables(staged_bundle.duckdb_path)
print("Registered DuckDB tables:", registered_tables)

sales_preview = query_duckdb(
    staged_bundle.duckdb_path,
    "SELECT region, COUNT(*) AS orders, SUM(revenue) AS total_revenue FROM synthetic_sales GROUP BY region ORDER BY region",
)
sales_preview

In [ ]:
hr_preview = query_duckdb(
    staged_bundle.duckdb_path,
    "SELECT d.department_name, COUNT(*) AS headcount, ROUND(AVG(e.base_salary_usd), 2) AS avg_salary "
    "FROM synthetic_hr__employees e JOIN synthetic_hr__departments d ON e.department_id = d.department_id "
    "GROUP BY d.department_name ORDER BY d.department_name",
)
print(hr_preview)

health_preview = query_duckdb(
    staged_bundle.duckdb_path,
    "SELECT department, COUNT(*) AS visits, ROUND(AVG(billed_amount_usd), 2) AS avg_billed "
    "FROM synthetic_healthcare GROUP BY department ORDER BY department",
)
health_preview